# Import ACLED data
- Author: Bryan Bravo
- Created: 2026-03-23
## Import Libraries

In [1]:
import os
import sys

os.chdir("C:/Users/bravo/OneDrive/bravo_projects/MLProject/StraitofHormuzResearch")
# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = 'C:/Program Files/Java/jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)


# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

import proj_vars

### Initialize Spark Session


In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


### Custom Functions

### Variables

In [3]:
end_date = proj_vars.end_date
end_date

'2026-03-22'

## Query

In [4]:
country_mapping = {
    'australia': 'AUS',
    'brazil': 'BRA',
    'canada': 'CAN',
    'china': 'CHN',
    # 'euro': 'EU',  # No IMF Data, must source from elsewhere.
    'france': 'FRA',
    'germany': 'DEU',
    'india': 'IND',
    'italy': 'ITA',
    'japan': 'JPN',
    'mexico': 'MEX',
    'south_korea': 'KOR',
    'russia': 'RUS',
    'south_africa': 'ZAF',
    'turkiye': 'TUR',
    'united_kingdom': 'GBR',
    'united_states': 'USA'
}


### Import ACLED: Number of political violence events by country-month-year
This includes all battle, explosions/remote violence, and violence against civilians events.
I'm currently importing from local environment, since dataset requires consent to import.

Check acleddata.com for updated excel file.

In [5]:
country_mapping = {
    'australia': 'AUS',
    'brazil': 'BRA',
    'canada': 'CAN',
    'china': 'CHN',
    # 'euro': 'EU',  # No IMF Data, must source from elsewhere.
    'france': 'FRA',
    'germany': 'DEU',
    'india': 'IND',
    'italy': 'ITA',
    'japan': 'JPN',
    'mexico': 'MEX',
    'south_korea': 'KOR',
    'russia': 'RUS',
    'south_africa': 'ZAF',
    'turkiye': 'TUR',
    'united_kingdom': 'GBR',
    'united_states': 'USA'
}

In [6]:
raw_data = pd.read_excel('import_datasets/number_of_political_violence_events_by_country-month-year_as-of-13Mar2026.xlsx',
                         header=0)
acled_df = raw_data.copy()
acled_df.columns = [col.lower() for col in acled_df.columns]

# subset for countries in the analysis
acled_df['country'] = acled_df['country'].str.lower().str.strip()
acled_df = acled_df[acled_df['country'].isin(country_name.replace('_', ' ') for country_name in country_mapping.keys())]
acled_df['country'] = acled_df['country'].str.replace(' ', '_')

# subset for year greater than or equal to 2006
acled_df = acled_df[acled_df['year'] >= 2006]

# remap month to integer
acled_df['month'] = acled_df['month'].map({
    mon: str(i+1) for i, mon in enumerate(['January', 'February', 'March', 'April', 'May', 'June',
                                    'July', 'August', 'September', 'October', 'November', 'December'])
    }).astype(int)



In [ ]:
# Convert to Spark DF
acled_df = spark.createDataFrame(acled_df)
acled_df.cache().count()

In [8]:
acled_df.orderBy('year', 'month', 'country').show()

+------------+-----+----+------+
|     country|month|year|events|
+------------+-----+----+------+
|south_africa|    1|2006|     1|
|south_africa|    2|2006|     2|
|south_africa|    3|2006|     0|
|south_africa|    4|2006|     2|
|south_africa|    5|2006|     1|
|south_africa|    6|2006|     1|
|south_africa|    7|2006|     1|
|south_africa|    8|2006|     2|
|south_africa|    9|2006|     2|
|south_africa|   10|2006|     0|
|south_africa|   11|2006|     0|
|south_africa|   12|2006|     1|
|south_africa|    1|2007|     2|
|south_africa|    2|2007|     5|
|south_africa|    3|2007|     5|
|south_africa|    4|2007|     2|
|south_africa|    5|2007|     1|
|south_africa|    6|2007|     4|
|south_africa|    7|2007|     1|
|south_africa|    8|2007|     7|
+------------+-----+----+------+
only showing top 20 rows

